# Dask arrays on one node

**Core notebook, about 15 minutes.** A Dask array looks like a NumPy array but is divided into chunks. Chunks are the units of scheduling and memory management.

In [ ]:
import os
import time
import numpy as np
import dask
import dask.array as da

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
side = 2_000 if test_mode else 6_000
rng = np.random.default_rng(2026)
array = rng.random((side, side), dtype=np.float64) + 0.1
print(f"Array shape: {array.shape}")
print(f"Array size: {array.nbytes / 1024**3:.2f} GiB")

## 1. Time the regular NumPy version

This expression creates intermediate arrays. Keep the result so we can verify the Dask calculation.

In [ ]:
started = time.perf_counter()
expected = array**2 + np.sin(array) * array * np.log(array)
numpy_time = time.perf_counter() - started
print(f"NumPy time: {numpy_time:.3f} s")

## 2. Choose chunks

A chunk is one piece of the array. Very small chunks create many tiny tasks for Dask to manage. Very large chunks may use too much memory or leave some workers with nothing to do.

In [ ]:
chunk_side = 1_000
dask_array = da.from_array(array, chunks=(chunk_side, chunk_side))
print("Chunks:", dask_array.chunks)
print("Blocks:", dask_array.numblocks)
print(f"Bytes per full chunk: {chunk_side**2 * array.dtype.itemsize / 1024**2:.1f} MiB")

In [ ]:
result_graph = (
    dask_array**2
    + da.sin(dask_array) * dask_array * da.log(dask_array)
)
result_graph

Nothing above has computed the output. `compute()` now schedules the blocks on a small thread pool.

In [ ]:
n_workers = min(4, os.cpu_count() or 1)
started = time.perf_counter()
actual = result_graph.compute(scheduler="threads", num_workers=n_workers)
dask_time = time.perf_counter() - started

np.testing.assert_allclose(actual, expected, rtol=1e-12)
print(f"Dask time with {n_workers} workers: {dask_time:.3f} s")

## Your turn: reason about chunk size

**10 minutes.** Try chunk sides of `250`, `1000`, and `3000` without changing the mathematical expression.

For each choice, record:

- Number of blocks.
- Approximate bytes per chunk.
- Compute time.
- One reason the choice may or may not work well for a larger array.

Do not expect Dask to beat NumPy for every array that already fits in memory. The lesson is how chunk size controls the number of tasks, memory use, and how work is shared.

In [ ]:
def measure_chunks(chunk_size):
    candidate = da.from_array(array, chunks=(chunk_size, chunk_size))
    graph = candidate**2 + da.sin(candidate) * candidate * da.log(candidate)
    started = time.perf_counter()
    computed = graph.compute(scheduler="threads", num_workers=n_workers)
    elapsed = time.perf_counter() - started
    np.testing.assert_allclose(computed, expected, rtol=1e-12)
    return candidate.numblocks, elapsed

if not test_mode:
    for candidate_size in (250, 1000, 3000):
        blocks, elapsed = measure_chunks(candidate_size)
        print(f"chunk={candidate_size:4d}, blocks={blocks}, time={elapsed:.3f} s")
else:
    print("Extended chunk sweep skipped in automated test mode.")

## Takeaway

A useful chunk fits in memory, gives workers enough pieces to share, and does not create too many tiny tasks. Inspect the task graph and chunks before calling `compute()`.